# Phase 3 training on Colab

Generated from `xBD_damage_to_carbon_cost.ipynb` (Sections 3b and 3c) by `tools/make_colab_notebook.py`. The model, losses and training loop below are **the same cells** as in the main notebook; only the data-loading cells differ, because here the crops are read from Google Drive instead of being cut from the raw images.

**Before running:** Runtime → Change runtime type → **T4 GPU**. Then Runtime → Run all, and approve the Google Drive prompt.

**It needs** these six files together in `My Drive/ENGG2112_Project/` (or in one subfolder of it): `crops_train_64px.npy`, `crops_tier3_64px.npy`, `crops_test_64px.npy` and their three `_meta.csv` files.

**It trains** every experiment in the table whose training data is listed in `COLAB_TRAIN_ON` and which has no result file in the Drive folder yet. Experiments already there are skipped, so it is safe to run again after a disconnect.

**It produces** three files per experiment in that same Drive folder (`p3_<name>_..._history.csv`, `..._test_predictions.csv`, `....weights.h5`). When done, download the new files starting with `p3_` into `xbd_project/outputs/` on the laptop.

In [ ]:
import shutil, time
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import keras
from keras import layers
from sklearn.metrics import f1_score
from IPython.display import display

COLAB_TRAIN_ON = ["train+tier3"]       # which experiments to train here (Set 1, "train", was done on the laptop)

try:                                   # on Colab: results go to Drive, crops are copied to the fast local disk
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_FOLDER = Path("/content/drive/MyDrive/ENGG2112_Project")
    # the crop files may sit directly in that folder or in a subfolder such as outputs/
    found = sorted(DRIVE_FOLDER.rglob("crops_train_64px.npy"))
    assert found, f"crops_train_64px.npy not found anywhere under {DRIVE_FOLDER}"
    OUTPUT_DIR = found[0].parent
    print("Using Drive folder:", OUTPUT_DIR)
    needed = [f"crops_{n}_64px{s}" for n in ("train", "tier3", "test") for s in (".npy", "_meta.csv")]
    missing = [f for f in needed if not (OUTPUT_DIR / f).exists()]
    assert not missing, f"missing from {OUTPUT_DIR}: {missing}"
    CROP_DIR = Path("/content/crops"); CROP_DIR.mkdir(exist_ok=True)
    for f in sorted(OUTPUT_DIR.glob("crops_*")):
        if not (CROP_DIR / f.name).exists() or (CROP_DIR / f.name).stat().st_size != f.stat().st_size:
            print("copying", f.name, round(f.stat().st_size / 1e9, 2), "GB"); shutil.copy(f, CROP_DIR / f.name)
    ON_COLAB = True
except ImportError:                    # anywhere else: use the local outputs folder for both
    OUTPUT_DIR = CROP_DIR = Path(__file__).resolve().parent / "outputs" if "__file__" in globals() else Path("./outputs")
    ON_COLAB = False

import tensorflow as tf
print("GPU:", tf.config.list_physical_devices("GPU") or "NONE - training will be very slow")

DAMAGE_CLASSES = ["no-damage", "minor-damage", "major-damage", "destroyed"]  # ordinal order
P3_CROP_SIZE = 64
VALIDATION_FRACTION = 0.10
baseline_cls_df = None                 # the baseline's predictions live in the main notebook only

In [ ]:
import psutil

def load_crops(name, into_memory):
    """Read one cached crop file. into_memory=True loads it all into RAM, which makes
    training much faster than fetching each batch from disk; False reads on demand."""
    meta = pd.read_csv(CROP_DIR / f"crops_{name}_{P3_CROP_SIZE}px_meta.csv")
    path = CROP_DIR / f"crops_{name}_{P3_CROP_SIZE}px.npy"
    if into_memory:
        free_gb = psutil.virtual_memory().available / 1e9
        if free_gb - path.stat().st_size / 1e9 < 2.5:      # keep 2.5 GB spare for training itself
            print(f"{name}: only {free_gb:.1f} GB RAM free, reading from disk instead (slower)")
            into_memory = False
    crops = np.load(path) if into_memory else np.load(path, mmap_mode="r")
    print(f"{name}: {crops.shape} {'in memory' if into_memory else 'on disk'}")
    return meta, crops

train_meta, train_crops = load_crops("train", into_memory=ON_COLAB)
tier3_meta, tier3_crops = load_crops("tier3", into_memory=ON_COLAB)
test_meta, test_crops = load_crops("test", into_memory=False)

# Train / validation split BY TILE (fixed seed, so it is the same every run)
tiles = np.sort(train_meta["file"].unique())
rng = np.random.default_rng(0)
val_tiles = set(rng.choice(tiles, size=int(len(tiles) * VALIDATION_FRACTION), replace=False))
train_meta["subset"] = np.where(train_meta["file"].isin(val_tiles), "validation", "train")
test_meta["subset"] = "test"
tier3_meta["subset"] = "tier3"   # all of tier3 is training data; validation stays inside the train split


print(train_meta["subset"].value_counts().to_dict())

In [ ]:
P3_TRAIN = True                  # on Colab: train every experiment that has no result yet
P3_BATCH_SIZE = 128
P3_WIDTH = 16                    # channels in the first encoder block
N_CLASSES = len(DAMAGE_CLASSES)

# One row per experiment.
#   train_on          which crop pools it learns from
#   samples_per_epoch random training buildings drawn each epoch (a fresh draw every epoch)
#   lr / lr_schedule  starting learning rate; "constant" keeps it, "cosine" lowers it smoothly to zero
#   seed              random seed (same seed = same starting weights and same sampling order)
#   encoder           "scratch" = the small CNN below; "efficientnet" = ImageNet-pretrained EfficientNetB0
P3_EXPERIMENTS = pd.DataFrame([
    # Set 1: train split only, small budget (laptop CPU, about an hour each)
    ("cnn_ce",                        "ce",    False, "train",       10,  40_000, 1e-3, "constant", 0, "scratch"),
    ("cnn_weighted",                  "ce",    True,  "train",       10,  40_000, 1e-3, "constant", 0, "scratch"),
    ("cnn_focal",                     "focal", True,  "train",       10,  40_000, 1e-3, "constant", 0, "scratch"),
    ("cnn_ordinal",                   "coral", True,  "train",       10,  40_000, 1e-3, "constant", 0, "scratch"),
    # Set 2: train + tier3, larger budget (Colab GPU)
    ("cnn_ce+tier3",                  "ce",    False, "train+tier3", 20, 150_000, 1e-3, "constant", 0, "scratch"),
    ("cnn_weighted+tier3",            "ce",    True,  "train+tier3", 20, 150_000, 1e-3, "constant", 0, "scratch"),
    ("cnn_focal+tier3",               "focal", True,  "train+tier3", 20, 150_000, 1e-3, "constant", 0, "scratch"),
    ("cnn_ordinal+tier3",             "coral", True,  "train+tier3", 20, 150_000, 1e-3, "constant", 0, "scratch"),
    # Set 3: the two best Set 2 recipes with a cosine learning-rate schedule, each with two seeds,
    # plus a pretrained encoder (Colab GPU)
    ("cnn_ce+tier3_cosine",           "ce",    False, "train+tier3", 20, 150_000, 1e-3, "cosine",   0, "scratch"),
    ("cnn_ce+tier3_cosine_seed1",     "ce",    False, "train+tier3", 20, 150_000, 1e-3, "cosine",   1, "scratch"),
    ("cnn_focal+tier3_cosine",        "focal", True,  "train+tier3", 20, 150_000, 1e-3, "cosine",   0, "scratch"),
    ("cnn_focal+tier3_cosine_seed1",  "focal", True,  "train+tier3", 20, 150_000, 1e-3, "cosine",   1, "scratch"),
    ("effnet_ce+tier3_cosine",        "ce",    False, "train+tier3", 20, 150_000, 3e-4, "cosine",   0, "efficientnet"),
    ("effnet_focal+tier3_cosine",     "focal", True,  "train+tier3", 20, 150_000, 3e-4, "cosine",   0, "efficientnet"),
], columns=["name", "loss", "class_weights", "train_on", "epochs", "samples_per_epoch", "lr", "lr_schedule", "seed", "encoder"])


# ----------------------------------------------------------------- the model
def build_encoder(norm="batch"):
    """Small CNN: 64x64x3 crop -> 128-number description of it.
    norm="batch" (Phase 3) normalises each layer using statistics of the whole batch;
    norm="group" (Layer 2) normalises each image on its own, see Section 4b for why."""
    x = crop = keras.Input((P3_CROP_SIZE, P3_CROP_SIZE, 3))
    for channels in [P3_WIDTH, P3_WIDTH * 2, P3_WIDTH * 4, P3_WIDTH * 8]:
        for _ in range(2):
            x = layers.Conv2D(channels, 3, padding="same", use_bias=False)(x)
            x = layers.BatchNormalization()(x) if norm == "batch" else layers.GroupNormalization(groups=8)(x)
            x = layers.ReLU()(x)
        x = layers.MaxPooling2D()(x)
    return keras.Model(crop, layers.GlobalAveragePooling2D()(x), name="encoder")


class OrdinalThresholds(layers.Layer):
    """CORAL output: takes ONE damage score and adds a separate learned offset for each
    of the 3 thresholds (>= minor, >= major, destroyed). Output > 0 means 'past that threshold'."""
    def build(self, input_shape):
        self.offsets = self.add_weight(shape=(N_CLASSES - 1,), initializer="zeros", name="offsets")
    def call(self, score):
        return score + self.offsets


def build_pretrained_encoder():
    """EfficientNetB0 pretrained on ImageNet, used as the encoder instead of the small CNN.
    Weights are downloaded by Keras from https://storage.googleapis.com/keras-applications/
    (efficientnetb0_notop.h5; Tan & Le 2019, 'EfficientNet', released with Keras Applications,
    Apache 2.0 licence). It expects pixel values 0-255 and does its own input scaling."""
    # (no custom name here: Keras builds the download file name from the model's own name)
    return keras.applications.EfficientNetB0(include_top=False, weights="imagenet", pooling="avg",
                                             input_shape=(P3_CROP_SIZE, P3_CROP_SIZE, 3))


def build_pair_classifier(ordinal=False, encoder="scratch"):
    pair = keras.Input((P3_CROP_SIZE, P3_CROP_SIZE, 6))          # before RGB + after RGB
    if encoder == "scratch":
        x = layers.Rescaling(1 / 255.0)(pair)
        shared = build_encoder()                                  # ONE encoder, used twice
    else:
        x = pair                                                  # EfficientNet scales its own input
        shared = build_pretrained_encoder()
    before, after = shared(x[..., :3]), shared(x[..., 3:])
    h = layers.Concatenate()([before, after, layers.Subtract()([after, before])])
    h = layers.Dense(128, activation="relu")(h)
    h = layers.Dropout(0.3)(h)
    if ordinal:
        out = OrdinalThresholds()(layers.Dense(1, use_bias=False)(h))   # 3 threshold scores
    else:
        out = layers.Dense(N_CLASSES)(h)                                 # 4 class scores
    return keras.Model(pair, out)


# ---------------------------------------------------------------- the losses
def cross_entropy(y_true, scores):
    return keras.losses.sparse_categorical_crossentropy(y_true, scores, from_logits=True)

def focal_loss(y_true, scores, gamma=2.0):
    """Cross-entropy scaled by (1 - probability given to the true class)^gamma:
    confident correct answers contribute almost nothing, mistakes contribute fully."""
    ce = cross_entropy(y_true, scores)
    return (1.0 - keras.ops.exp(-ce)) ** gamma * ce

def coral_loss(y_true, scores):
    """Three yes/no questions per building: is the true class above threshold 0, 1, 2?
    e.g. major-damage (class 2) -> [yes, yes, no]. Loss = average binary cross-entropy."""
    y = keras.ops.cast(keras.ops.reshape(y_true, (-1, 1)), "float32")
    passed = keras.ops.cast(y > keras.ops.arange(N_CLASSES - 1, dtype="float32"), "float32")
    return keras.ops.mean(keras.ops.binary_crossentropy(passed, scores, from_logits=True), axis=-1)

P3_LOSSES = {"ce": cross_entropy, "focal": focal_loss, "coral": coral_loss}

def scores_to_class(scores, ordinal):
    """Model output -> class index 0..3. Ordinal: count how many thresholds were passed."""
    return (scores > 0).sum(axis=1) if ordinal else scores.argmax(axis=1)


# ------------------------------------------------------------------ the data
class CropBatches(keras.utils.PyDataset):
    """Feeds batches of crop pairs straight from the cached file on disk."""
    def __init__(self, crops, labels, indices, sample_weights=None, samples_per_epoch=None, training=False, seed=0):
        super().__init__()
        self.crops, self.labels, self.indices = crops, labels, np.asarray(indices)
        self.sample_weights, self.training = sample_weights, training
        self.n = len(self.indices) if samples_per_epoch is None else min(int(samples_per_epoch), len(self.indices))
        self.rng = np.random.default_rng(seed)
        self.order = self.rng.permutation(self.indices) if training else self.indices

    def __len__(self):
        return int(np.ceil(self.n / P3_BATCH_SIZE))

    def __getitem__(self, i):
        idx = np.sort(self.order[i * P3_BATCH_SIZE:min((i + 1) * P3_BATCH_SIZE, self.n)])
        x = np.asarray(self.crops[idx])
        if self.training:   # augmentation: the before and after crops are flipped/rotated together
            flip = self.rng.random(len(idx)) < 0.5
            x[flip] = x[flip][:, :, ::-1]
            x = np.rot90(x, k=int(self.rng.integers(4)), axes=(1, 2))
        y = self.labels[idx]
        w = np.ones(len(idx), np.float32) if self.sample_weights is None else self.sample_weights[idx]
        return np.ascontiguousarray(x), y, w

    def on_epoch_end(self):
        if self.training:   # fresh random draw of training buildings for the next epoch
            self.order = self.rng.permutation(self.indices)


class CropStore:
    """Several cached crop files presented as one long array (nothing is copied into memory)."""
    def __init__(self, arrays):
        self.arrays = arrays
        self.starts = np.cumsum([0] + [len(a) for a in arrays])
    def __len__(self):
        return int(self.starts[-1])
    def __getitem__(self, idx):   # idx: sorted array of positions in the combined array
        parts = [a[idx[(idx >= lo) & (idx < hi)] - lo] for a, lo, hi in zip(self.arrays, self.starts[:-1], self.starts[1:])]
        return np.concatenate(parts)


# The training pool: train split followed by tier3, addressed by one index
pool_meta = pd.concat([train_meta, tier3_meta], ignore_index=True)
pool_crops = CropStore([train_crops, tier3_crops])
pool_labels = pool_meta["label"].to_numpy()
test_labels = test_meta["label"].to_numpy()
val_idx = np.flatnonzero((pool_meta["has_crop"] & (pool_meta["subset"] == "validation")).to_numpy())

def training_indices(train_on):
    """Positions in the pool that an experiment may learn from, e.g. "train" or "train+tier3"."""
    return np.flatnonzero((pool_meta["has_crop"] & pool_meta["subset"].isin(train_on.split("+"))).to_numpy())

def balanced_class_weights(indices):
    """total / (number of classes x count of that class): rare classes weigh more."""
    counts = np.bincount(pool_labels[indices], minlength=N_CLASSES)
    return len(indices) / (N_CLASSES * counts)

weights_table = {}
for train_on in P3_EXPERIMENTS["train_on"].unique():
    idx = training_indices(train_on)
    weights_table[(train_on, "training buildings")] = np.bincount(pool_labels[idx], minlength=N_CLASSES)
    weights_table[(train_on, "class weight")] = balanced_class_weights(idx).round(2)
display(pd.DataFrame(weights_table, index=DAMAGE_CLASSES))

example_model = build_pair_classifier()
print(f"Pair classifier: {example_model.count_params():,} parameters "
      f"(encoder {example_model.get_layer('encoder').count_params():,}, shared between before and after)")
P3_EXPERIMENTS

In [ ]:
# Train only the chosen sets here; anything else in the table is left for the laptop / already done.
P3_EXPERIMENTS = P3_EXPERIMENTS[P3_EXPERIMENTS["train_on"].isin(COLAB_TRAIN_ON)].reset_index(drop=True)
done = [n for n, e, s in zip(P3_EXPERIMENTS["name"], P3_EXPERIMENTS["epochs"], P3_EXPERIMENTS["samples_per_epoch"])
        if (OUTPUT_DIR / f"p3_{n}_{e}ep_{s}spe_test_predictions.csv").exists()]
print("Already on Drive (will be skipped):", done)
print("To train now:", [n for n in P3_EXPERIMENTS["name"] if n not in done])

In [ ]:
import time

def run_experiment(exp):
    """Train one row of P3_EXPERIMENTS (or load it from cache). Returns (history, test predictions) or None."""
    tag = f"p3_{exp.name}_{exp.epochs}ep_{exp.samples_per_epoch}spe"
    weights_path = OUTPUT_DIR / f"{tag}.weights.h5"
    history_path = OUTPUT_DIR / f"{tag}_history.csv"
    preds_path = OUTPUT_DIR / f"{tag}_test_predictions.csv"
    if history_path.exists() and preds_path.exists():
        return pd.read_csv(history_path), pd.read_csv(preds_path)
    if not P3_TRAIN:
        return None

    ordinal = exp.loss == "coral"
    keras.utils.set_random_seed(exp.seed)
    model = build_pair_classifier(ordinal, exp.encoder)
    train_idx = training_indices(exp.train_on)
    weights = balanced_class_weights(train_idx)[pool_labels].astype(np.float32) if exp.class_weights else None
    train_batches = CropBatches(pool_crops, pool_labels, train_idx, weights, exp.samples_per_epoch, training=True, seed=exp.seed)
    val_batches = CropBatches(pool_crops, pool_labels, val_idx)
    if exp.lr_schedule == "cosine":   # learning rate falls smoothly from exp.lr to zero over the whole run
        learning_rate = keras.optimizers.schedules.CosineDecay(exp.lr, decay_steps=exp.epochs * len(train_batches))
    else:
        learning_rate = exp.lr
    model.compile(optimizer=keras.optimizers.Adam(learning_rate), loss=P3_LOSSES[exp.loss])

    history, best_f1, start = [], -1.0, time.time()
    for epoch in range(1, exp.epochs + 1):
        fit = model.fit(train_batches, epochs=1, verbose=0)
        val_pred = scores_to_class(model.predict(val_batches, verbose=0), ordinal)
        val_f1 = f1_score(pool_labels[val_idx], val_pred, average="macro")
        history.append({"experiment": exp.name, "epoch": epoch, "train_loss": fit.history["loss"][0],
                        "val_macro_f1": val_f1, "minutes": (time.time() - start) / 60})
        if val_f1 > best_f1:                       # keep the best epoch's weights
            best_f1 = val_f1
            model.save_weights(weights_path)
    model.load_weights(weights_path)

    test_idx = np.flatnonzero(test_meta["has_crop"].to_numpy())
    test_pred = scores_to_class(model.predict(CropBatches(test_crops, test_labels, test_idx), verbose=0), ordinal)
    preds = test_meta.iloc[test_idx][["file", "uid", "disaster", "damage_class"]].rename(columns={"damage_class": "true_class"})
    preds["pred_class"] = [DAMAGE_CLASSES[i] for i in test_pred]
    history = pd.DataFrame(history)
    history.to_csv(history_path, index=False); preds.to_csv(preds_path, index=False)
    return history, preds


# model_predictions: every model's test predictions, used again in Section 7b
model_predictions = {"baseline (pretrained)": baseline_cls_df}
p3_histories = []
for exp in P3_EXPERIMENTS.itertuples(index=False):
    result = run_experiment(exp)
    if result is None:
        print(f"{exp.name}: not trained yet (no cached result and P3_TRAIN is False)")
        continue
    p3_histories.append(result[0]); model_predictions[exp.name] = result[1]
    best = result[0].loc[result[0]["val_macro_f1"].idxmax()]
    print(f"{exp.name}: best validation macro F1 {best['val_macro_f1']:.3f} at epoch {int(best['epoch'])} "
          f"({result[0]['minutes'].max():.0f} min of training)")

In [ ]:
summary = []
for h in p3_histories:
    name = h["experiment"].iloc[0]; p = model_predictions[name]
    per_class = f1_score(p["true_class"], p["pred_class"], labels=DAMAGE_CLASSES, average=None)
    summary.append({"experiment": name, "best val macro F1": h["val_macro_f1"].max(), "test macro F1": per_class.mean(),
                    **{f"test F1 {c}": f for c, f in zip(DAMAGE_CLASSES, per_class)}, "minutes": h["minutes"].max()})
display(pd.DataFrame(summary).set_index("experiment").round(3))

fig, ax = plt.subplots(figsize=(11, 4.5))
for h in p3_histories:
    ax.plot(h["epoch"], h["val_macro_f1"], marker="o", markersize=3, label=h["experiment"].iloc[0])
ax.set_xlabel("epoch"); ax.set_ylabel("validation macro F1"); ax.legend(fontsize=7, ncol=2); plt.show()

print("Files to download into xbd_project/outputs/ on the laptop (skip any you already have):")
for f in sorted(OUTPUT_DIR.glob("p3_*")): print("  ", f.name)